# 502: Decline Statistics by Scenario Type

This notebook calculates decline statistics (delta_warming = peak - 2100) for three scenario groups:
1. NZCO2 scenarios (zeroed after net-zero CO2)
2. NZKyoto scenarios (zeroed after net-zero Kyoto GHG)
3. Base scenarios (no NZCO2 or NZKyoto suffix)

In [24]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

dotenv.load_dotenv()

True

In [25]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

# Input metrics file
MANIFEST_ID = "2026-05-26_14-58"
METRICS_FILE = OUTPUT_FOLDER / f"500_gsat_metrics_{MANIFEST_ID}.csv"

In [36]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE)
print(f"Total rows: {len(metrics_df)}")

# Filter for ALL magicc_flag only
metrics_all = metrics_df[metrics_df["magicc_flag"] == "ALL"].copy()
print(f"ALL magicc_flag rows: {len(metrics_all)}")

Total rows: 9600
ALL magicc_flag rows: 2400


In [27]:
def calculate_scenario_stats(df):
    """
    Calculate per-scenario statistics for delta_warming.
    
    Returns DataFrame with median, 34-67 percentile, and 5-95 percentile per scenario.
    """
    stats = df.groupby(["model", "scenario"]).agg(
        n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
        median=pd.NamedAgg(column="delta_warming", aggfunc="median"),
        p5=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 5)),
        p34=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 34)),
        p67=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 67)),
        p95=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 95)),
    ).reset_index()
    return stats


def calculate_group_summary(scenario_stats):
    """
    Calculate summary statistics across all scenarios in a group.
    
    Returns dict with:
    - median of medians
    - 34-67 percentile range of combined 34-67 values
    - 5-95 percentile range of combined 5-95 values
    """
    return {
        "n_scenarios": len(scenario_stats),
        "median_of_medians": scenario_stats["median"].median(),
        "p34_of_p34": scenario_stats["p34"].quantile(0.34),
        "p67_of_p67": scenario_stats["p67"].quantile(0.67),
        "p5_of_p5": scenario_stats["p5"].quantile(0.05),
        "p95_of_p95": scenario_stats["p95"].quantile(0.95),
    }

## 1. NZCO2 Scenarios

In [28]:
# Filter for NZCO2 scenarios
nzco2_df = metrics_all[metrics_all["scenario"].str.contains("NZCO2", na=False)].copy()
print(f"NZCO2 rows: {len(nzco2_df)}")
print(f"NZCO2 unique scenarios: {nzco2_df['scenario'].nunique()}")

NZCO2 rows: 0
NZCO2 unique scenarios: 0


In [38]:
metrics_df["scenario"].unique()

array(['ENGAGE-NPi2020-400f-lowBECCS',
       'DeepElectrification-SSP2-HighRenewables-900',
       'LowEnergyDemand (1.3/IPCC-AR6)',
       'ENGAGE-NPi2020-400f-lowBECCS_NZCO2', 'SusDev-SDP-1000',
       'DeepElectrification-SSP2-HighRenewables-900_NZCO2',
       'LowEnergyDemand (1.3/IPCC-AR6)_NZCO2', 'SusDev-SDP-1000_NZCO2'],
      dtype=object)

In [29]:
# Per-scenario statistics
nzco2_stats = calculate_scenario_stats(nzco2_df)
nzco2_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95


In [30]:
# Group summary
nzco2_summary = calculate_group_summary(nzco2_stats)
print("NZCO2 Group Summary:")
for k, v in nzco2_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

NZCO2 Group Summary:
  n_scenarios: 0
  median_of_medians: nan
  p34_of_p34: nan
  p67_of_p67: nan
  p5_of_p5: nan
  p95_of_p95: nan


## 2. NZKyoto Scenarios

In [31]:
# Filter for NZKyoto scenarios
nzkyoto_df = metrics_all[metrics_all["scenario"].str.contains("NZKyoto", na=False)].copy()
print(f"NZKyoto rows: {len(nzkyoto_df)}")
print(f"NZKyoto unique scenarios: {nzkyoto_df['scenario'].nunique()}")

NZKyoto rows: 0
NZKyoto unique scenarios: 0


In [32]:
# Per-scenario statistics
nzkyoto_stats = calculate_scenario_stats(nzkyoto_df)
nzkyoto_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95


In [18]:
# Group summary
nzkyoto_summary = calculate_group_summary(nzkyoto_stats)
print("NZKyoto Group Summary:")
for k, v in nzkyoto_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

NZKyoto Group Summary:
  n_scenarios: 0
  median_of_medians: nan
  p34_of_p34: nan
  p67_of_p67: nan
  p5_of_p5: nan
  p95_of_p95: nan


## 3. Base Scenarios (no NZCO2 or NZKyoto)

In [19]:
# Filter for base scenarios (no NZCO2 or NZKyoto suffix)
base_df = metrics_all[
    ~metrics_all["scenario"].str.contains("NZCO2", na=False) &
    ~metrics_all["scenario"].str.contains("NZKyoto", na=False)
].copy()
print(f"Base scenario rows: {len(base_df)}")
print(f"Base unique scenarios: {base_df['scenario'].nunique()}")

Base scenario rows: 2400
Base unique scenarios: 4


In [20]:
# Per-scenario statistics
base_stats = calculate_scenario_stats(base_df)
base_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95
0,COFFEE 1.1,ENGAGE-NPi2020-400f-lowBECCS,600,0.2200,0.06690,0.18700,0.25600,0.35805
1,MESSAGEix-GLOBIOM 1.0,LowEnergyDemand (1.3/IPCC-AR6),600,0.2555,0.05300,0.20566,0.29033,0.38205
2,REMIND-MAgPIE 2.1-4.2,SusDev-SDP-1000,600,0.2900,0.09790,0.25066,0.32666,0.42400
3,REMIND-MAgPIE 2.1-4.3,DeepElectrification-SSP2-HighRenewables-900,600,0.1080,0.00795,0.07400,0.14100,0.23115


In [21]:
# Group summary
base_summary = calculate_group_summary(base_stats)
print("Base Scenarios Group Summary:")
for k, v in base_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

Base Scenarios Group Summary:
  n_scenarios: 4
  median_of_medians: 0.238
  p34_of_p34: 0.187
  p67_of_p67: 0.291
  p5_of_p5: 0.015
  p95_of_p95: 0.418


## Combined Summary

In [22]:
# Create combined summary table
summary_df = pd.DataFrame([
    {"group": "NZCO2", **nzco2_summary},
    {"group": "NZKyoto", **nzkyoto_summary},
    {"group": "Base", **base_summary},
])

# Round numeric columns
numeric_cols = [c for c in summary_df.columns if c not in ["group", "n_scenarios"]]
summary_df[numeric_cols] = summary_df[numeric_cols].round(3)

summary_df

,group,n_scenarios,median_of_medians,p34_of_p34,p67_of_p67,p5_of_p5,p95_of_p95
0,NZCO2,0,NaN,NaN,NaN,NaN,NaN
1,NZKyoto,0,NaN,NaN,NaN,NaN,NaN
2,Base,4,0.238,0.187,0.291,0.015,0.418


## Save Results

In [23]:
# Save per-scenario statistics
nzco2_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_nzco2_{MANIFEST_ID}.csv", index=False)
nzkyoto_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_nzkyoto_{MANIFEST_ID}.csv", index=False)
base_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_base_{MANIFEST_ID}.csv", index=False)

# Save combined summary
summary_df.to_csv(OUTPUT_FOLDER / f"502_decline_summary_{MANIFEST_ID}.csv", index=False)

print(f"Results saved to {OUTPUT_FOLDER}")

Results saved to /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output
